In [1]:
import yadisk # https://pypi.org/project/yadisk/
import os
from dotenv import load_dotenv

import cv2 # https://docs.opencv.org/4.x/d1/dc5/tutorial_background_subtraction.html
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib import colors

import pickle
import json 
import os

from tqdm import tqdm
import time
from pprint import pprint

from collections import Counter

In [2]:
load_dotenv()
TOKEN = os.getenv('DEBUG_TOKEN')
client = yadisk.Client(token=TOKEN)

with client:
    # Проверяет, валиден ли токен
    print(client.check_token())

    # Выводит содержимое "disk:/SLR Project"
    files = list(client.listdir("disk:/SLR Project")) # disk:/SLR_Project_Cuts
    print(len(files))

True
482


In [4]:
def cut_the_video(path_on_disk,  path_to_store, path_to_save, video_name,
                  window_size=11, overlap=5, new_fps=2):
    TOKEN = os.getenv('DEBUG_TOKEN')
    client = yadisk.Client(token=TOKEN)
    client.download(path_on_disk, path_to_store) # скачиваем файл
    cap = cv2.VideoCapture(path_to_store) # читаем файл


    size = (int(cap.get(3)), int(cap.get(4))) # ширина, высота видео
    fps = cap.get(cv2.CAP_PROP_FPS) # frames per second
    fpw = int(fps*window_size) # frames per window 
    fpo = int(fps*overlap) # frames per overlap

    window_frames = [] # тут кадры с исходным fps
    wfi = []
    i = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        window_frames.append(frame)
        wfi.append(i)
        
        if len(window_frames) >= fpw: # пора записывать текущее окно и обрезать
            new_video_name = f"{path_to_save+'.'.join(video_name.split('.')[:-1])}_{int(i-fpw)}_{int(i)}_{int((i-fpw)/fps)}_{int(i/fps)}.mp4"
            out = cv2.VideoWriter(new_video_name, 
                         cv2.VideoWriter_fourcc(*'mp4v'), #MP4V 
                         new_fps, 
                         size)
            for nfi, new_frame in enumerate(window_frames):
                if nfi % new_fps == 0:
                    out.write(new_frame)  
            out.release()
            #client.upload(new_video_name, f"disk:/SLR_Project_Cuts/{new_video_name}")
            #os.remove(new_video_name)
            #time.sleep(5)
            window_frames = window_frames[fpo:] # редактируем window_framеs
            wfi = wfi[fpo:]

        i += 1
    cap.release() # закрываем исходный файл
    os.remove(path_to_store) # удаляем исходный файл

In [ ]:
# Для тестов
video_name = "#УСЛЫШЬМЕНЯ 25 сентября. Премьера по всей России. На жестовом языке, с субтитрами.mp4"
path_on_disk = 'disk:/SLR Project/'+ video_name
path_to_store = './' + video_name
path_to_save = 'files2upload/'

cut_the_video(path_on_disk, path_to_store, path_to_save, video_name,
                window_size=11, overlap=5, new_fps=3)

In [9]:
# Все видео
no_subs_set = set() # видео без субтитров нас не интересуют
with open('../EDA/no_subs.json', 'r', encoding='utf-8') as f:
    no_subs = json.load(f)
    for no_sub in no_subs:
        no_subs_set.add(no_sub['vid_path'].replace('\"', ''))

path_to_save = 'files2upload2/'

for file in tqdm(files[10:15]):
    video_name = file['path'].split('/')[-1]
    if video_name not in no_subs_set:
        path_on_disk = file['path']
        path_to_store = './' + video_name
        cut_the_video(path_on_disk, path_to_store, path_to_save, video_name,
                        window_size=11, overlap=5, new_fps=3)


100%|██████████| 5/5 [09:02<00:00, 108.54s/it]


In [8]:
for file in files:
    print(file['path'])

disk:/SLR Project/#УСЛЫШЬМЕНЯ 25 сентября. Премьера по всей России. На жестовом языке, с субтитрами.mp4
disk:/SLR Project/#УСЛЫШЬМЕНЯ Глухая Москва. С субтитрами.mp4
disk:/SLR Project/#УСЛЫШЬМЕНЯ Предпремьерное. На жестовом языке, с субтитрами.mp4
disk:/SLR Project/#УСЛЫШЬМЕНЯ Премьера состоялась в кинотеатре Космос. На жестовом языке, с субтитрами.mp4
disk:/SLR Project/14-ый ПЛЕНЭР R+Я 2016. На жестовом языке, с субтитрами.mp4
disk:/SLR Project/3 часть. Жизнь и приключения глухих и культмассовика Нины Марийсовой. С субтитрами.mp4
disk:/SLR Project/60-ти летие фотографа Прикащикова. С субтитрами.mp4
disk:/SLR Project/85 лет журналу В едином строю. С субтитрами.mp4
disk:/SLR Project/95-летие Сурдлимпийского движения.mp4
disk:/SLR Project/COVID 19 Глазами наших соотечественников за рубежом. С субтитрами.mp4
disk:/SLR Project/Curling  Керлинг в Мадезимо. Сурдлимпиада 2019.mp4
disk:/SLR Project/Cоглашение между ВОГ и Ассоциацией ветеранов спорта глухих. С субтитрами.mp4
disk:/SLR Project/I